In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

# %matplotlib qt
%matplotlib inline
import matplotlib.pyplot as plt

from pathlib import Path
import spikeinterface.extractors as se
import spikeinterface.full as si
from neuropy.core.session.init_from_raw_data import windows_to_wsl_path_if_needed, find_first_file_rglob

from spikeinterface_pipeline import BapunSessionConfig, CurationConfig, resolve_session_paths, load_bapun_recording, load_phy_sorting, load_spykingcircus_sorting, run_phy_curation_pipeline, compute_qm_labels


In [ ]:
## Bapun Format:
# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day1Openfield') # Linux
# basedir = Path('/nfs/turbo/umms-kdiba/Data/Bapun/RatS/Day1Openfield') # Greatlakes
# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day1Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day1Openfield") # Apogee WSL
# # n_channels: int = 200
# n_channels: int = 195
# dat_file_sampling_rate: int = 30000
# eeg_sampling_rate: int = 1250
# basename: str = 'RatS-Day1Openfield'
# excluded_data_datetimes = ['2020-11-25_10-24-24', '2020-11-25_15-06-02']


# # basedir = Path('/home/halechr/FastData/Bapun/RatS/Day4Openfield') # Linux
# basedir = Path('/nfs/turbo/umms-kdiba/Data/Bapun/RatS/Day4Openfield') # Greatlakes
# # basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day4Openfield')) # Windows
# # basedir = Path("/mnt/w/Data/Bapun/RatS/Day4Openfield") # Apogee WSL
# n_channels: int = 195
# dat_file_sampling_rate: int = 30000
# eeg_sampling_rate: int = 1250
# basename: str = 'RatS-Day4Openfield'
# excluded_data_datetimes = []


# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day4Openfield') # Linux
# basedir = Path('/nfs/turbo/umms-kdiba/Bapun/RatS/Day5TwoNovel') # Greatlakes Turbo
basedir = Path('/scratch/kdiba_root/kdiba99/halechr/Data/Bapun/RatS/Day5TwoNovel') # Greatlakes

# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day4Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day4Openfield") # Apogee WSL
n_channels: int = 195
dat_file_sampling_rate: int = 30000
eeg_sampling_rate: int = 1250
basename: str = 'RatS-Day5TwoNovel-2020-12-04_07-55-09'
excluded_data_datetimes = []


In [ ]:
session = BapunSessionConfig(basedir=basedir, basename=basename, n_channels=n_channels, dat_file_sampling_rate=dat_file_sampling_rate, eeg_sampling_rate=eeg_sampling_rate, excluded_data_datetimes=excluded_data_datetimes)
paths = resolve_session_paths(session)
xml_path = paths.xml_path
concatenated_dat_file = paths.concatenated_dat_file
print(f'xml_path: {xml_path}')
print(f"concatenated_dat_file: {concatenated_dat_file.as_posix()}")
assert concatenated_dat_file.exists(), f"concatenated_dat_file: {concatenated_dat_file} does not exist."
paths

In [ ]:
spiking_circus_loaded = load_spykingcircus_sorting(paths)
spiking_circus_loaded

In [ ]:
phy_gui_dir = paths.phy_gui_dir
assert phy_gui_dir.exists(), f"phy_gui_dir: {phy_gui_dir} does not exist!"


In [ ]:
sorting = load_phy_sorting(paths)
sorting


In [ ]:
n_channels

In [ ]:
recording, recording_filtered = load_bapun_recording(session, paths)
recording


In [ ]:
recording_filtered


In [ ]:
sorting = spiking_circus_loaded

job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s")

sorting_analyzer_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}-sorting_analyzer").resolve())
sorting_analyzer = si.create_sorting_analyzer(sorting, recording_filtered, format="binary_folder", folder=sorting_analyzer_folder.as_posix())
sorting_analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
sorting_analyzer.compute("waveforms", **job_kwargs)
sorting_analyzer.compute("templates", **job_kwargs)
sorting_analyzer.compute("noise_levels")
sorting_analyzer.compute("unit_locations", method="monopolar_triangulation")
sorting_analyzer.compute("isi_histograms")
sorting_analyzer.compute("correlograms", window_ms=100, bin_ms=5.)
sorting_analyzer.compute("principal_components", n_components=3, mode='by_channel_global', whiten=True, **job_kwargs)
sorting_analyzer.compute("quality_metrics", metric_names=["snr", "firing_rate"])
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("spike_amplitudes", **job_kwargs)

In [ ]:
from spikeinterface.sorters import installed_sorters, run_sorter

installed_sorters() # ['kilosort4', 'lupin', 'simple', 'spykingcircus2', 'tridesclous2']

sorting_outputs_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("SORTING").resolve()) # , f"{basename}-sorting_analyzer"
sorting_outputs_folder.mkdir(exist_ok=True)
print(f'sorting_outputs_folder: {sorting_outputs_folder}')


In [ ]:

# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"), remove_existing_folder=True, delete_output_folder=True)


In [ ]:
# run Tridesclous
TDC_Output_Path: Path = sorting_outputs_folder.joinpath("folder_TDC")
sorting_TDC = run_sorter(sorter_name="tridesclous", recording=recording_filtered, folder=TDC_Output_Path)
sorting_TDC

In [ ]:
TDC_Final_Sorting_Output_Path: Path = TDC_Output_Path.joinpath('tridescloud_sorting_output')
sorting_TDC.save(folder=TDC_Final_Sorting_Output_Path)


In [ ]:
sorting_TDC.sorting_info

In [ ]:
# run Kilosort4
sorting_KS2_5 = run_sorter(sorter_name="kilosort4", recording=recording, folder=sorting_outputs_folder.joinpath("folder_KS4"))

In [ ]:
sorting = run_sorter(sorter_name="spykingcircus2", recording=recording, folder="/folder_SC2tmp/folder")

In [ ]:
# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"))
sorting_SC2

In [ ]:
sorting_SC2


# Quality Metrics Tutorial

After spike sorting, you might want to validate the 'goodness' of the sorted units. This can be done using the
:code:`qualitymetrics` submodule, which computes several quality metrics of the sorted units.


In [ ]:
import spikeinterface.core as si
from spikeinterface.metrics import (
    compute_snrs,
    compute_presence_ratios,
    compute_isi_violations,
)

First, let's generate a simulated recording and sorting



In [ ]:
si.set_global_job_kwargs(n_jobs=11, chunk_duration="1s", progress_bar=False)
# sorting = sorting_TDC
# sorting_analyzer = sorting_TDC.to_shared_memory_sorting()

sorting_analyzer = si.create_sorting_analyzer(sorting=sorting, recording=recording_filtered)
sorting_analyzer.compute(['noise_levels','random_spikes','waveforms','templates','spike_locations','spike_amplitudes','correlograms','principal_components','quality_metrics','template_metrics'])


In [ ]:
sorting_analyzer.compute('template_metrics', include_multi_channel_metrics=True) 

# recording, sorting = si.generate_ground_truth_recording()
# print(recording)
# print(sorting)

In [ ]:
# also clear in-memory registry if needed
sorting_analyzer.extensions.pop("spike_amplitudes", None)
sorting_analyzer.extensions.pop("templates", None)


In [ ]:
import spikeinterface as si
si.set_global_job_kwargs(n_jobs=8, chunk_duration="1s", progress_bar=True)

sorting_analyzer.compute("spike_amplitudes")

ext = sorting_analyzer.get_extension("spike_amplitudes")
print(ext is not None)
print(ext.get_data().shape)   # should be (n_spikes,) or similar

In [ ]:
print(sorting_analyzer.has_recording())
print(sorting_analyzer.get_loaded_extension_names())

sorting_analyzer.compute({
    # "random_spikes": {},
    # "waveforms": {},
    "templates": {},
    # "noise_levels": {},
})

In [ ]:
all_metric_names = list(sorting_analyzer.get_extension('quality_metrics').get_data().keys()) + list(sorting_analyzer.get_extension('template_metrics').get_data().keys())
print(set(model.feature_names_in_).issubset(set(all_metric_names)))

In [ ]:
sorting_analyzer.get_extension('templates')

In [ ]:
sorting_analyzer.compute(['noise_levels','random_spikes','waveforms','templates','spike_locations','correlograms','principal_components','quality_metrics','template_metrics']) # ,'spike_amplitudes'
sorting_analyzer.compute('template_metrics', include_multi_channel_metrics=True) 

## Create SortingAnalyzer

For quality metrics we need first to create a :code:`SortingAnalyzer`.



The :code:`spikeinterface.qualitymetrics` submodule has a set of functions that allow users to compute
metrics in a compact and easy way. To compute a single metric, one can simply run one of the
quality metric functions as shown below. Each function has a variety of adjustable parameters that can be tuned.



In [ ]:
## INPUTS: sorting_analyzer
presence_ratios = compute_presence_ratios(sorting_analyzer)
print(presence_ratios)
isi_violation_ratio, isi_violations_count = compute_isi_violations(sorting_analyzer)
print(isi_violation_ratio)
snrs = compute_snrs(sorting_analyzer)
print(snrs)

In [ ]:
labels = sc.model_based_label_units(
    sorting_analyzer = sorting_analyzer,
    repo_id = "SpikeInterface/toy_tetrode_model",
    trusted = ['numpy.dtype']
)

print(labels)

To compute more than one metric at once, we can use the :code:`SortingAnalyzer.compute("quality_metrics")`
function and indicate which metrics we want to compute. Then we can retrieve the results using the :code:`get_data()`
method as a ``pandas.DataFrame``.



In [ ]:
metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=["presence_ratio", "snr", "amplitude_cutoff"],
    metric_params={
        "presence_ratio": {"bin_duration_s": 2.0},
    }
)
metrics = metrics_ext.get_data()
print(metrics)

Some metrics are based on the principal component scores, so the extension
must be computed before. For instance:



In [ ]:
sorting_analyzer.compute("principal_components", n_components=5, mode="by_channel_global", whiten=True)

metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=[
        "mahalanobis",
        "d_prime",
    ],
)
metrics = metrics_ext.get_data()
print(metrics)

# Trained Models

In [ ]:
from spikeinterface.curation import model_based_label_units

labels_and_probabilities = model_based_label_units(
    sorting_analyzer = sorting_analyzer,
    repo_id = "SpikeInterface/toy_tetrode_model",
    trust_model = True
)

In [ ]:
GOOD_UNIT_STRATEGY: str = "sua_high_conf"
PROB_THRESHOLD_HIGH: float = 0.8
curation = CurationConfig(strategy="sua_high_conf", prob_high=PROB_THRESHOLD_HIGH, analyzer_overwrite="always", require_cluster_info=False)


In [ ]:
result = run_phy_curation_pipeline(session, curation, patch_pandas_compat=True)
sorting = result.sorting
sorting_analyzer = result.sorting_analyzer
all_labels = result.all_labels
comparison = result.comparison
good_units = result.good_units
sorting_curated = result.sorting_curated
print(all_labels["prediction"].value_counts())
print(comparison.groupby(["phy_label", "prediction"]).size())


In [ ]:
sorting

In [ ]:
metrics_df = sorting_analyzer.get_extension("quality_metrics").get_data()
template_df = sorting_analyzer.get_extension("template_metrics").get_data()

out_dir = sorting_analyzer_folder / "exports"
out_dir.mkdir(exist_ok=True)
metrics_df.to_csv(out_dir / "quality_metrics.csv")
template_df.to_csv(out_dir / "template_metrics.csv")

In [ ]:
metrics_df

In [ ]:
template_df

## Write out labels to Phy

In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).



/nfs/turbo/umms-kdiba/Bapun/RatS/Day1Openfield/spyk-circ
/nfs/turbo/umms-kdiba/Bapun/RatS/Day4Openfield/spyk-circ
/nfs/turbo/umms-kdiba/Bapun/RatS/Day5TwoNovel/spyk-circ


# Load sorting_analyzer directly from disk

In [ ]:
from pathlib import Path
import spikeinterface.full as si
import spikeinterface.curation as sc

sorting_analyzer_folder = Path("/scratch/kdiba_root/kdiba99/halechr/Data/Bapun/RatS/Day5TwoNovel/SORTING/folder_SC2_sorting_analyzer")
sorting_analyzer = si.load_sorting_analyzer(sorting_analyzer_folder.as_posix())

# Sanity check  all should print True; if so, do NOT call .compute()
for ext in ["quality_metrics", "template_metrics", "waveforms", "templates", "principal_components"]:
    print(ext, sorting_analyzer.has_extension(ext))

print(sorting_analyzer)

In [ ]:
si.set_global_job_kwargs(n_jobs=11, chunk_duration="1s", progress_bar=False)

sorting_analyzer.compute({
    "random_spikes": {},
    "waveforms": {},
    "templates": {},
    "noise_levels": {},
    "spike_amplitudes": {},
    "spike_locations": {},
    "principal_components": {"n_components": 5, "mode": "by_channel_local"},
})
# Compute as many quality metrics as possible (default), including PCA ones
sorting_analyzer.compute("quality_metrics")

In [ ]:
si.set_global_job_kwargs(n_jobs=11, chunk_duration="1s", progress_bar=False)

# Define the extensions you want, along with their parameters
extensions_to_run = {
    "random_spikes": {"max_spikes_per_unit": 500},
    "waveforms": {"ms_before": 1.0, "ms_after": 2.0},
    "templates": {"operators": ["average", "median"]},
    "noise_levels": {},
    "spike_amplitudes": {},
    "spike_locations": {},    
    "principal_components": {"n_components": 3, "mode": "by_channel_local"},
    'correlograms': {},
    'template_metrics': {'include_multi_channel_metrics': True},
    # "quality_metrics": dict(    metric_names=["presence_ratio", "snr", "amplitude_cutoff", "mahalanobis", "d_prime", 'amplitude_median', 'num_spikes'], # , 'rp_contamination', 'drift_ptp'
    #                             metric_params={
    #                                 "presence_ratio": {"bin_duration_s": 2.0},
    #                             }),
    "quality_metrics": {},
}

extensions_to_run = extensions_to_run | {k:{} for k in ['spike_locations','spike_amplitudes','correlograms']}
extensions_to_run

active_extensions_to_run = {ext_name:ext_params for ext_name, ext_params in extensions_to_run.items() if (not sorting_analyzer.has_extension(ext_name))}
active_extensions_to_run


In [ ]:
if len(active_extensions_to_run) > 0:
	sorting_analyzer.compute(active_extensions_to_run)


In [ ]:
sorting_analyzer.compute("principal_components", n_components=5, mode="by_channel_global", whiten=True)

In [ ]:
metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    # metric_names=[
    #     "mahalanobis",
    #     "d_prime",
    # ],
)
metrics = metrics_ext.get_data()
print(metrics)

In [ ]:
# Compute as many quality metrics as possible (default), including PCA ones
sorting_analyzer.compute("quality_metrics")

In [ ]:
import spikeinterface as si
import spikeinterface.qualitymetrics as sqm
import spikeinterface.curation as sc

si.set_global_job_kwargs(n_jobs=11, chunk_duration="1s", progress_bar=True)

assert sorting_analyzer.has_recording(), "UnitRefine needs raw recording attached to analyzer"

sorting_analyzer.compute({
    "random_spikes": {},
    "waveforms": {},
    "templates": {},
    "noise_levels": {},
    "spike_amplitudes": {},
    "spike_locations": {},
    "principal_components": {"n_components": 5, "mode": "by_channel_local"},
})

# Compute as many quality metrics as possible (default), including PCA ones
sorting_analyzer.compute("quality_metrics")

qm = sorting_analyzer.get_extension("quality_metrics").get_data()
required = {
    "drift_mad", "snr", "firing_range", "amplitude_cutoff", "nn_hit_rate",
    "isi_violations_count", "sync_spike_2", "isi_violations_ratio", "sync_spike_4",
    "nn_miss_rate", "drift_std", "rp_contamination", "num_spikes", "amplitude_cv_range",
    "presence_ratio", "firing_rate", "amplitude_cv_median", "sliding_rp_violation",
    "silhouette", "sync_spike_8", "rp_violations", "amplitude_median", "drift_ptp",
}
print("still missing:", required - set(qm.columns))
print(qm.isna().sum()[list(required & set(qm.columns))])

In [ ]:
# Option A: single noise/neural classifier
labels = sc.model_based_label_units(
    sorting_analyzer=sorting_analyzer,
    repo_id="SpikeInterface/UnitRefine_noise_neural_classifier",
    trust_model=True,
)
labels.head()

In [ ]:
# Option B: full two-stage cascade (what si-curate-sorter uses internally)
from spikeinterface_pipeline.curation import run_unitrefine_two_stage
all_labels, analyzer_neural, noise_labels = run_unitrefine_two_stage(sorting_analyzer)
all_labels.head()